# Oscillation Detection

Analysis for the selected simulation run. The data loading, metrics, tables, and
figures are visible below so they can be reviewed or customized.

In [ ]:
from pathlib import Path

DATA_DIR = r""
RUN_DIR = Path(DATA_DIR) if DATA_DIR else None

## Analysis definitions

These cells contain the small data-loading and event-matching helpers used below.
They keep the report self-contained in the notebook.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from IPython.display import display

NAVY = "#1A204C"
MBLUE = "#085A9B"
GREEN = "#1D772A"
ORANGE = "#FAA449"
RED = "#C00000"


def _load_wiring(run_dir: Path) -> dict:
    path = Path(run_dir) / "wiring.json"
    if not path.exists():
        raise FileNotFoundError(f"wiring.json not found in {run_dir}")
    return json.loads(path.read_text(encoding="utf-8"))


def _component_params(wiring: dict, component_type: str) -> dict:
    for component in wiring.get("components", []):
        if component.get("type") == component_type:
            return component.get("parameters") or {}
    return {}

In [ ]:
def _load_events(run_dir: Path) -> pd.DataFrame:
    path = run_dir / "outputs" / "events.feather"
    if not path.exists():
        raise FileNotFoundError(f"events.feather not found in {run_dir / 'outputs'}")
    df = pd.read_feather(path)
    if "event_id" in df.columns:
        df = df.drop_duplicates(subset="event_id", keep="last").reset_index(drop=True)
    return df


def _ground_truth(run_dir: Path) -> list[dict] | None:
    wiring = _load_wiring(run_dir)
    params = _component_params(wiring, "ODComponent")
    raw = params.get("ground_truth_json")
    if not raw:
        return None
    try:
        truth = json.loads(raw) if isinstance(raw, str) else raw
        return truth if isinstance(truth, list) and truth else None
    except (ValueError, TypeError):
        return None


def _signal(run_dir: Path):
    """Measured input signal from the Player CSV: (t_seconds, freq_dev_mHz, label).

    The build dir carries a copy of the component data, so the plot works on any
    machine the run was made on; falls back to OEDISI_COMPONENTS for older runs.
    """
    try:
        wiring = _load_wiring(run_dir)
        # Current OEDI-SI templates use PlayerComponent. Keep Player as a
        # compatibility fallback for reports generated from older run wiring.
        params = _component_params(wiring, "PlayerComponent")
        if not params:
            params = _component_params(wiring, "Player")
    except Exception:  # noqa: BLE001
        return None
    fname = str(params.get("filename") or "").replace("\\", "/")
    if not fname:
        return None
    base = fname.rsplit("/", 1)[-1]
    cands = [p for p in (run_dir / "build").rglob(base) if p.is_file()]
    if not cands:
        comp_root = os.environ.get("OEDISI_COMPONENTS")
        if comp_root and "/components/" in fname:
            root = Path(comp_root).expanduser().resolve()
            parts = fname.split("/components/", 1)[1].split("/")
            if parts and parts[0] == "ornl-od":
                parts[0] = "ornl-od-prony"
            c = root.joinpath(*parts).resolve()
            try:
                c.relative_to(root)
            except ValueError:
                c = None
            if c is not None and c.exists():
                cands = [c]
    if not cands:
        return None
    try:
        sig = pd.read_csv(cands[0])
        tcol = sig.columns[0]
        units = [c for c in sig.columns if c != tcol]
        if not units:
            return None
        try:
            tt = pd.to_datetime(sig[tcol])
            t = (tt - tt.iloc[0]).dt.total_seconds().to_numpy(dtype=float)
        except (ValueError, TypeError):
            step = float(params.get("run_freq_time_step") or 1.0)
            t = np.arange(len(sig)) * step
        y = pd.to_numeric(sig[units[0]], errors="coerce").to_numpy(dtype=float)
        n_steps = int(params.get("number_of_timesteps") or 0)
        if 0 < n_steps < len(t):
            t, y = t[:n_steps], y[:n_steps]
        dec = max(1, len(t) // 50000)
        t, y = t[::dec], y[::dec]
        dev_mhz = (y - np.nanmedian(y)) * 1000.0
        label = units[0].replace("_", " ")
        return t, dev_mhz, label
    except Exception:  # noqa: BLE001
        return None


def _match_pairs(df: pd.DataFrame, f0, truth):
    """Nearest-start matching (30 s tolerance), same rules as the metrics table."""
    if not truth or f0 is None:
        return [], set(), 0, 0
    used: set[int] = set()
    pairs: list[tuple[int, dict, int | None]] = []
    n = len(df)
    for k, tr in enumerate(truth):
        tf = tr.get("frequency_hz")
        ts = tr.get("start_time_sec")
        best_i, best_d = None, None
        for i in range(n):
            if i in used:
                continue
            if ts is not None and "start_time_sec" in df.columns:
                d = abs(float(df["start_time_sec"][i]) - float(ts))
                if d > 30.0:
                    continue
            elif tf:
                d = abs(float(f0[i]) - float(tf))
            else:
                continue
            if best_d is None or d < best_d:
                best_i, best_d = i, d
        if best_i is not None:
            used.add(best_i)
        pairs.append((k, tr, best_i))
    missed = sum(1 for _, _, i in pairs if i is None)
    fa = n - len(used)
    return pairs, used, missed, fa

## Run data

In [ ]:
events = pd.DataFrame()
n_events = 0
dominant_frequency = None
dominant_damping = None
ground_truth = None
matched_pairs = []
matched_detections = set()
missed_events = 0
false_alarms = 0
measured_signal = None

if RUN_DIR is None:
    print("Run the simulation and open its Notebook to see the analysis.")
else:
    events = _load_events(RUN_DIR)
    n_events = len(events)
    if n_events == 0:
        print("No oscillation events were detected in this run.")
    else:
        dominant_frequency = events.get("mode_0_freq_hz")
        dominant_damping = events.get("mode_0_damping")
        ground_truth = _ground_truth(RUN_DIR)
        matched_pairs, matched_detections, missed_events, false_alarms = _match_pairs(
            events, dominant_frequency, ground_truth
        )
        measured_signal = _signal(RUN_DIR)

## Detection timeline from the measured signal

In [ ]:
if RUN_DIR is not None and n_events > 0 and measured_signal is not None:
    time_seconds, deviation_mhz, unit_label = measured_signal
    fig, ax = plt.subplots(figsize=(10.8, 4.2))
    ax.plot(time_seconds, deviation_mhz, color=MBLUE, lw=0.7,
            label=f"Measured frequency ({unit_label})")
    amplitude = float(np.nanmax(np.abs(deviation_mhz))) if len(deviation_mhz) else 1.0
    limit = max(amplitude * 1.15, 1e-6)
    ax.set_ylim(-limit, limit)
    if ground_truth:
        for truth_index, truth_event in enumerate(ground_truth):
            truth_start = truth_event.get("start_time_sec")
            if truth_start is None:
                continue
            truth_start = float(truth_start)
            ax.axvline(truth_start, color="#666666", ls="--", lw=1.0)
            ax.axvspan(truth_start, truth_start + 60.0, color=ORANGE, alpha=0.12)
            ax.text(truth_start + 30.0, limit * 0.87, f"E{truth_index + 1}",
                    color=ORANGE, fontweight="bold", ha="center", fontsize=10)
    if "start_time_sec" in events.columns:
        ax.plot(events["start_time_sec"], np.zeros(n_events), "v", color=RED,
                ms=10, zorder=5, ls="none", label="detection")
    if ground_truth:
        status = (f"{len(matched_detections)}/{len(ground_truth)} detected, "
                  f"{false_alarms} false alarms, {missed_events} missed")
    else:
        status = f"{n_events} event{'s' if n_events != 1 else ''} detected"
    ax.set_title("Detection timeline — measured signal · true onsets (dashed) · "
                 f"red = detection · {status}", loc="left", fontsize=11,
                 color=NAVY, fontweight="bold")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("freq deviation (mHz)")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(axis="y", color="#EDF0F4", lw=0.8)
    display(fig)
    plt.close(fig)

## Detected event timeline

In [ ]:
if (RUN_DIR is not None and n_events > 0
        and "start_time_sec" in events.columns and dominant_frequency is not None):
    fig, ax = plt.subplots(figsize=(8.6, 3.6))
    colors = [RED if dominant_damping is not None and value < 0 else GREEN
              for value in (dominant_damping if dominant_damping is not None else [0] * n_events)]
    ax.stem(events["start_time_sec"], dominant_frequency, basefmt=" ",
            linefmt="-", markerfmt="o")
    event_ids = events.get("event_id", range(1, n_events + 1))
    for start, frequency, color, event_id in zip(
            events["start_time_sec"], dominant_frequency, colors, event_ids):
        ax.plot([start], [frequency], "o", color=color, ms=9, zorder=5)
        ax.annotate(f"E{int(event_id)}", (start, frequency), textcoords="offset points",
                    xytext=(6, 6), fontsize=9, color=NAVY, fontweight="bold")
    ax.set_xlabel("event start time (s)")
    ax.set_ylabel("dominant frequency (Hz)")
    ax.set_title("Detection timeline — when events occurred and at what frequency "
                 "(red = growing)", loc="left", fontsize=11, color=NAVY,
                 fontweight="bold")
    ax.grid(axis="y", color="#EDF0F4", lw=0.8)
    display(fig)
    plt.close(fig)

## Mode frequency compared with ground truth

In [ ]:
if RUN_DIR is not None and n_events > 0 and ground_truth and dominant_frequency is not None:
    frequency_rows = [(truth_index, truth_event, event_index)
                      for truth_index, truth_event, event_index in matched_pairs
                      if event_index is not None and truth_event.get("frequency_hz")]
    if frequency_rows:
        fig, ax = plt.subplots(figsize=(7.6, 3.6))
        x = np.arange(len(frequency_rows))
        truth_values = [float(event["frequency_hz"]) for _, event, _ in frequency_rows]
        detected_values = [float(dominant_frequency[index]) for _, _, index in frequency_rows]
        ax.bar(x - 0.16, truth_values, width=0.3, color=NAVY, label="true")
        ax.bar(x + 0.16, detected_values, width=0.3, color=GREEN, label="detected")
        ax.set_xlim(-0.6, max(len(frequency_rows), 4) - 0.4)
        for xpos, detected in zip(x, detected_values):
            ax.text(xpos + 0.16, detected, f"{detected:.4g}", ha="center",
                    va="bottom", fontsize=9, color=GREEN, fontweight="bold")
        accuracy = [max(0.0, 1 - abs(detected - truth) / truth) * 100
                    for truth, detected in zip(truth_values, detected_values)]
        ax.set_xticks(x)
        ax.set_xticklabels([f"E{index + 1}" for index, _, _ in frequency_rows])
        ax.set_ylabel("frequency (Hz)")
        ax.margins(y=0.15)
        ax.set_title(f"Mode frequency vs truth · {float(np.mean(accuracy)):.0f}% match",
                     loc="left", fontsize=11, color=NAVY, fontweight="bold")
        ax.legend(loc="upper right", fontsize=9)
        ax.grid(axis="y", color="#EDF0F4", lw=0.8)
        display(fig)
        plt.close(fig)

## Damping sign and stability

In [ ]:
if RUN_DIR is not None and n_events > 0 and dominant_damping is not None:
    fig, ax = plt.subplots(figsize=(7.8, 3.6))
    x = np.arange(n_events)
    damping_values = [float(value) for value in dominant_damping]
    colors = [RED if value < 0 else GREEN for value in damping_values]
    ax.bar(x, damping_values, color=colors, width=0.4)
    ax.set_xlim(-0.6, max(n_events, 4) - 0.4)
    span = max(abs(min(damping_values)), abs(max(damping_values)), 1e-9)
    for xpos, value in zip(x, damping_values):
        ax.text(xpos, value + (span * 0.04 if value >= 0 else -span * 0.04),
                f"{value:+.3f}", ha="center", va="bottom" if value >= 0 else "top",
                fontsize=9, fontweight="bold", color=GREEN if value >= 0 else RED)
    ax.axhline(0, color=NAVY, lw=1.0)
    labels = [f"E{int(event_id)}" for event_id in events.get(
        "event_id", range(1, n_events + 1))]
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("damping ratio")
    ax.margins(y=0.25)
    growing = [label for label, value in zip(labels, damping_values) if value < 0]
    note = f"{'/'.join(growing)} GROWING = alarm" if growing else "all decaying (safe)"
    ax.set_title(f"Damping sign → stability · {note}", loc="left", fontsize=11,
                 color=NAVY, fontweight="bold")
    ax.legend(handles=[Patch(color=GREEN, label="+ decaying (safe)"),
                       Patch(color=RED, label="− growing (alarm)")],
              loc="upper right", fontsize=8)
    display(fig)
    plt.close(fig)

## Ground-truth metrics

In [ ]:
metrics_summary = pd.DataFrame()
ground_truth_metrics = pd.DataFrame()

if RUN_DIR is not None and n_events > 0:
    if ground_truth:
        def _number(event_index: int, column: str):
            if column not in events.columns:
                return None
            value = events[column][event_index]
            try:
                value = float(value)
            except (TypeError, ValueError):
                return None
            return value if np.isfinite(value) else None

        metric_rows = []
        used_event_indices: set[int] = set()
        latency_values: list[float] = []
        frequency_accuracy_values: list[float] = []
        amplitude_accuracy_values: list[float] = []
        damping_accuracy_values: list[float] = []
        for _, truth_event, event_index in matched_pairs:
            tf = truth_event.get("frequency_hz")
            ts = truth_event.get("start_time_sec")
            ta = truth_event.get("amplitude")
            tz = truth_event.get("damping_ratio")
            if event_index is None:
                metric_rows.append([ts if ts is not None else "—", tf, "—", "—",
                                    "—", "—", "MISSED"])
                continue
            used_event_indices.add(event_index)
            fd = float(dominant_frequency[event_index])
            facc = max(0.0, 1 - abs(fd - float(tf)) / float(tf)) * 100 if tf else None
            latency = None
            reporting_time = _number(event_index, "reporting_time_sec")
            if ts is not None and reporting_time is not None and tf:
                latency = (reporting_time - float(ts)) * float(tf)
            aacc = None
            detected_start = _number(event_index, "start_time_sec")
            detected_amplitude = _number(event_index, "mode_0_amp")
            if None not in (ts, ta, tz, detected_start, detected_amplitude) and tf:
                omega_d = 2 * np.pi * float(tf)
                sigma = float(tz) * omega_d / np.sqrt(1 - float(tz) ** 2)
                equivalent_amplitude = detected_amplitude / np.exp(
                    -sigma * (detected_start - float(ts)))
                aacc = max(0.0, 1 - abs(equivalent_amplitude - float(ta)) / float(ta)) * 100
            dacc = None
            detected_damping = _number(event_index, "mode_0_damping")
            if tz is not None and detected_damping is not None and float(tz) != 0:
                dacc = max(0.0, 1 - abs(detected_damping - float(tz)) / abs(float(tz))) * 100
            for value, values in ((latency, latency_values), (facc, frequency_accuracy_values),
                                  (aacc, amplitude_accuracy_values),
                                  (dacc, damping_accuracy_values)):
                if value is not None:
                    values.append(value)
            metric_rows.append([
                ts if ts is not None else "—", tf, f"{fd:.4g}",
                f"{facc:.1f} %" if facc is not None else "—",
                f"{aacc:.1f} %" if aacc is not None else "—",
                f"{dacc:.1f} %" if dacc is not None else "—",
                f"{latency:.2f} cyc" if latency is not None else "—",
            ])
        for event_index in range(n_events):
            if event_index not in used_event_indices:
                metric_rows.append(["—", "—", f"{float(dominant_frequency[event_index]):.4g}",
                                    "—", "—", "—", "FALSE ALARM"])
        missed_count = sum(1 for row in metric_rows if row[-1] == "MISSED")
        false_alarm_count = n_events - len(used_event_indices)

        def _mean(values: list[float], pattern: str) -> str:
            return pattern.format(sum(values) / len(values)) if values else "—"

        metrics_summary = pd.DataFrame([
            ("truth events", str(len(ground_truth))),
            ("matched", str(len(used_event_indices))),
            ("missed", str(missed_count)),
            ("false alarms", str(false_alarm_count)),
            ("mean latency", _mean(latency_values, "{:.2f} cyc")),
            ("freq accuracy", _mean(frequency_accuracy_values, "{:.1f} %")),
            ("amplitude accuracy", _mean(amplitude_accuracy_values, "{:.1f} %")),
            ("damping accuracy", _mean(damping_accuracy_values, "{:.1f} %")),
        ], columns=["metric", "value"])
        ground_truth_metrics = pd.DataFrame(metric_rows, columns=[
            "true start (s)", "true freq (Hz)", "detected freq (Hz)",
            "freq accuracy", "amplitude accuracy", "damping accuracy",
            "detection latency",
        ])
        display(metrics_summary)
        display(ground_truth_metrics)
    else:
        print("No ground truth configured for this run — real field events report detection "
              "and frequency characterization only (full accuracy scoring needs known truth).")

## Detected events (dominant mode)

In [ ]:
detected_events_table = pd.DataFrame()

if RUN_DIR is not None and n_events > 0:
    displayed_columns = [column for column in [
        "event_id", "time", "start_time_sec", "reporting_time_sec", "n_voted_units",
        "mode_0_freq_hz", "mode_0_damping", "mode_0_amp",
    ] if column in events.columns]
    displayed_rows = []
    for _, event in events[displayed_columns].iterrows():
        displayed_rows.append([
            f"{event[column]:.4g}" if isinstance(event[column], float) else str(event[column])
            for column in displayed_columns
        ])
    detected_events_table = pd.DataFrame(displayed_rows, columns=displayed_columns)
    display(detected_events_table)